# Patent Emerging Radar — Full Kaggle Pipeline

Runs the complete pipeline on a T4 GPU:
1. Load embeddings from `embeddings-patent` dataset
2. Cluster with RAPIDS cuML GPU UMAP + HDBSCAN
3. Feature engineering (18 features per cluster per quarter)
4. Train GRU + LSTM for 2yr and 3yr forecasting (growth-delta target)
5. 5yr linear extrapolation
6. Generate leaderboards
7. Zip everything → `pipeline_output.zip`

**Input dataset:** `embeddings-patent` (attach it to this notebook)

**Output:** Download `pipeline_output.zip` from the Output tab, then:
```bash
cd /home/jin/Documents/GITHUB/patent-emerging-radar
unzip ~/Downloads/pipeline_output.zip -d data/processed/
```

In [ ]:
import subprocess, sys

# cuML (GPU UMAP + HDBSCAN) is pre-installed on Kaggle T4 via RAPIDS
try:
    import cuml
    print(f'cuML {cuml.__version__} — GPU clustering enabled')
    GPU_MODE = True
except ImportError:
    print('cuML not found — falling back to CPU (umap-learn + hdbscan)')
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'umap-learn', 'hdbscan', '-q'])
    GPU_MODE = False

import tensorflow as tf
print(f'TensorFlow {tf.__version__}')
print(f'GPUs: {tf.config.list_physical_devices("GPU")}')

In [ ]:
import json, pathlib, pickle, warnings, zipfile, datetime
from collections import defaultdict

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import spearmanr
from tqdm.auto import tqdm
import joblib

warnings.filterwarnings('ignore')

# ── Paths ──────────────────────────────────────────────────────────────────────
KAGGLE_INPUT = pathlib.Path('/kaggle/input')
OUT_DIR      = pathlib.Path('/kaggle/working')

# Search recursively — dataset may be nested under /kaggle/input/datasets/USER/NAME/
DATASET_DIR = None
for p in KAGGLE_INPUT.rglob('embeddings.zip'):
    DATASET_DIR = p.parent
    print(f'Found embeddings.zip at: {p}')
    break
if DATASET_DIR is None:
    for p in KAGGLE_INPUT.rglob('*.npz'):
        DATASET_DIR = p.parent
        print(f'Found .npz files at: {DATASET_DIR}')
        break
if DATASET_DIR is None:
    all_files = [str(p) for p in KAGGLE_INPUT.rglob('*') if p.is_file()][:30]
    raise FileNotFoundError(
        f'No embeddings.zip or .npz files found anywhere under /kaggle/input/\n'
        f'Files found: {all_files}\n'
        'Attach the embeddings-patent dataset to this notebook.'
    )

EMBED_DIR = OUT_DIR / 'embeddings'
EMBED_DIR.mkdir(exist_ok=True)

print(f'Dataset : {DATASET_DIR}')
print(f'Output  : {OUT_DIR}')


In [ ]:
# ── Extract embeddings ─────────────────────────────────────────────────────────
npz_files = sorted(DATASET_DIR.glob('*.npz'))
if npz_files:
    EMBED_DIR = DATASET_DIR
    print(f'Found {len(npz_files)} .npz files directly in dataset.')
else:
    zip_path = DATASET_DIR / 'embeddings.zip'
    if not zip_path.exists():
        zips = list(DATASET_DIR.glob('*.zip'))
        if not zips:
            raise FileNotFoundError(
                f'No .npz files or .zip found in {DATASET_DIR}\n'
                f'Contents: {list(DATASET_DIR.iterdir())}'
            )
        zip_path = zips[0]
        print(f'Using zip: {zip_path.name}')
    with zipfile.ZipFile(zip_path) as zf:
        members = zf.infolist()
        print(f'Extracting {zip_path.name} ({zip_path.stat().st_size/1e9:.1f} GB) ...')
        for member in tqdm(members, desc='Extracting', unit='file'):
            zf.extract(member, EMBED_DIR)
    npz_files = sorted(EMBED_DIR.glob('*.npz'))
    print(f'Extracted {len(npz_files)} files.')

available_years = sorted(int(f.stem) for f in EMBED_DIR.glob('*.npz'))
print(f'Years: {available_years[0]}–{available_years[-1]}  ({len(available_years)} years)')


In [ ]:
# ── Full config ────────────────────────────────────────────────────────────────

# Clustering
WINDOW_YEARS   = 2
UMAP_NEIGHBORS = 50
UMAP_DIMS      = 10
HDBSCAN_MIN    = 50
CENTROID_SIM   = 0.85
CLUSTER_YEARS  = list(range(2002, 2026))  # use all available embeddings (2002–2025)
MIN_ACTIVE_Q   = 16

# Forecasting
WINDOW   = 8          # lookback quarters
HORIZONS = [8, 12]   # 2yr, 3yr model horizons
BATCH    = 64
EPOCHS   = 80
LR       = 3e-4

TRAIN_END = (2017, 4)
VAL_END   = (2020, 4)
TEST_END  = {8: (2023, 4), 12: (2022, 4)}  # per-horizon: data_end - H quarters

PRIMARY_H = HORIZONS[0]  # 2yr — used for val display

print('Config ready.')
print(f'Cluster years : {CLUSTER_YEARS[0]}–{CLUSTER_YEARS[-1]}')
print(f'Horizons      : {[f"{h//4}yr" for h in HORIZONS]}')

---
## Part 1 — Clustering

In [ ]:
# ── Load helpers ───────────────────────────────────────────────────────────────
def load_year(year):
    f = EMBED_DIR / f'{year}.npz'
    if not f.exists(): return None
    d = np.load(f, allow_pickle=True)
    return d['ids'], d['dates'], d['titles'], d['embeddings'].astype(np.float32)

def cosine_sim(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))

def get_quarter(date_str):
    y, m = int(str(date_str)[:4]), int(str(date_str)[5:7])
    return y, (m - 1) // 3 + 1

available = [y for y in CLUSTER_YEARS if (EMBED_DIR / f'{y}.npz').exists()]
print(f'{len(available)} years ready: {available[:3]} … {available[-3:]}')

In [ ]:
# ── cluster_window: GPU (cuML) with CPU fallback ───────────────────────────────
if GPU_MODE:
    from cuml.manifold import UMAP as cuUMAP
    from cuml.cluster.hdbscan import HDBSCAN as cuHDBSCAN

    def cluster_window(yrs):
        parts = [p for p in (load_year(y) for y in yrs) if p is not None]
        if not parts: return None
        ids    = np.concatenate([p[0] for p in parts])
        dates  = np.concatenate([p[1] for p in parts])
        titles = np.concatenate([p[2] for p in parts])
        embs   = np.concatenate([p[3] for p in parts])

        reducer   = cuUMAP(n_neighbors=UMAP_NEIGHBORS, n_components=UMAP_DIMS,
                           metric='cosine', random_state=42)
        reduced   = reducer.fit_transform(embs)

        clusterer = cuHDBSCAN(min_cluster_size=HDBSCAN_MIN,
                              metric='euclidean', cluster_selection_method='eom')
        labels    = clusterer.fit_predict(reduced)

        if hasattr(labels, 'values_host'): labels = labels.values_host
        elif hasattr(labels, 'get'):       labels = labels.get()
        else:                              labels = np.array(labels)

        n_cl  = len(set(labels)) - (1 if -1 in labels else 0)
        noise = (labels == -1).mean() * 100

        centroids = {}
        for c in set(labels):
            if c == -1: continue
            mask = labels == c
            cent = embs[mask].mean(axis=0)
            centroids[c] = cent / (np.linalg.norm(cent) + 1e-9)

        print(f'  {yrs}: {len(embs):,} patents → {n_cl} clusters  noise={noise:.1f}%')
        return dict(ids=ids, dates=dates, titles=titles,
                    labels=labels, centroids=centroids, n_clusters=n_cl)
else:
    import umap as umap_lib
    import hdbscan as hdbscan_lib

    def cluster_window(yrs):
        parts = [p for p in (load_year(y) for y in yrs) if p is not None]
        if not parts: return None
        ids    = np.concatenate([p[0] for p in parts])
        dates  = np.concatenate([p[1] for p in parts])
        titles = np.concatenate([p[2] for p in parts])
        embs   = np.concatenate([p[3] for p in parts])

        reducer   = umap_lib.UMAP(n_neighbors=UMAP_NEIGHBORS, n_components=UMAP_DIMS,
                                  metric='cosine', random_state=42, low_memory=True)
        reduced   = reducer.fit_transform(embs)

        clusterer = hdbscan_lib.HDBSCAN(min_cluster_size=HDBSCAN_MIN,
                                        metric='euclidean', cluster_selection_method='eom')
        labels    = clusterer.fit_predict(reduced)
        n_cl      = len(set(labels)) - (1 if -1 in labels else 0)
        noise     = (labels == -1).mean() * 100

        centroids = {}
        for c in set(labels):
            if c == -1: continue
            mask = labels == c
            cent = embs[mask].mean(axis=0)
            centroids[c] = cent / (np.linalg.norm(cent) + 1e-9)

        print(f'  {yrs}: {len(embs):,} patents → {n_cl} clusters  noise={noise:.1f}%')
        return dict(ids=ids, dates=dates, titles=titles,
                    labels=labels, centroids=centroids, n_clusters=n_cl)

print(f'cluster_window ready  (GPU={GPU_MODE})')

In [ ]:
# ── Full rolling-window run + cross-window stitching ──────────────────────────
cluster_series    = defaultdict(lambda: defaultdict(int))
cluster_centroids = {}
cluster_titles    = defaultdict(list)
stability_log     = []
next_gid          = 0

windows = [available[i:i+WINDOW_YEARS] for i in range(len(available) - WINDOW_YEARS + 1)]

for window in tqdm(windows, desc='Clustering windows', unit='window'):
    res = cluster_window(window)
    if res is None: continue
    stitched, born = 0, 0
    local_to_global = {}
    for loc, cent in res['centroids'].items():
        best_gid, best_s = -1, 0.0
        for gid, gcent in cluster_centroids.items():
            s = cosine_sim(cent, gcent)
            if s > best_s: best_s, best_gid = s, gid
        if best_s >= CENTROID_SIM:
            local_to_global[loc] = best_gid
            cluster_centroids[best_gid] = cent
            stitched += 1
        else:
            local_to_global[loc] = next_gid
            cluster_centroids[next_gid] = cent
            next_gid += 1
            born += 1
    stability_log.append({'window': str(window), 'stitched': stitched,
                          'born': born, 'total': res['n_clusters']})
    for pid, date, title, lbl in zip(res['ids'], res['dates'], res['titles'], res['labels']):
        if lbl == -1 or lbl not in local_to_global: continue
        gid = local_to_global[lbl]
        cluster_series[gid][get_quarter(str(date))] += 1
        if len(cluster_titles[gid]) < 25:
            cluster_titles[gid].append(str(title))

print(f'Total global clusters : {next_gid}')
print(f'With >= {MIN_ACTIVE_Q} quarters : {sum(1 for s in cluster_series.values() if len(s) >= MIN_ACTIVE_Q)}')


In [ ]:
# ── Build cluster panel ────────────────────────────────────────────────────────
panel_rows = []
for gid, ts in cluster_series.items():
    if len(ts) < MIN_ACTIVE_Q: continue
    for (y, q), cnt in ts.items():
        panel_rows.append({'cluster_id': gid, 'year': y, 'quarter': q,
                           'date': f'{y}-{(q-1)*3+1:02d}-01', 'count': cnt})

panel = pd.DataFrame(panel_rows)
all_yq = sorted({(r.year, r.quarter) for _, r in panel.iterrows()})
full_rows = []
for gid in panel.cluster_id.unique():
    sub = panel[panel.cluster_id == gid].set_index(['year', 'quarter'])['count']
    for y, q in all_yq:
        full_rows.append({'cluster_id': gid, 'year': y, 'quarter': q,
                          'date': f'{y}-{(q-1)*3+1:02d}-01', 'count': sub.get((y, q), 0)})

panel_full = (pd.DataFrame(full_rows)
              .sort_values(['cluster_id', 'date'])
              .reset_index(drop=True))

panel_full.to_csv(OUT_DIR / 'cluster_panel.csv', index=False)
print(f'Panel: {panel_full.cluster_id.nunique()} clusters x {len(all_yq)} quarters')

In [ ]:
# ── Auto-label clusters (TF-IDF + NMF) ────────────────────────────────────────
cluster_labels = {}
active_ids = set(panel_full.cluster_id.unique())

for gid in active_ids:
    titles_list = cluster_titles.get(gid, [])
    if not titles_list:
        cluster_labels[gid] = f'cluster_{gid}'
        continue
    corpus = [' '.join(titles_list)]
    try:
        vec = TfidfVectorizer(max_features=40, stop_words='english', ngram_range=(1, 2))
        X   = vec.fit_transform(corpus)
        nmf = NMF(n_components=1, random_state=42).fit(X)
        top_idx = nmf.components_[0].argsort()[::-1][:5]
        terms   = [vec.get_feature_names_out()[i] for i in top_idx]
        cluster_labels[gid] = ', '.join(terms)
    except Exception:
        cluster_labels[gid] = f'cluster_{gid}'

labels_df = pd.DataFrame([{'cluster_id': k, 'auto_label': v}
                           for k, v in cluster_labels.items()])
labels_df.to_csv(OUT_DIR / 'cluster_labels.csv', index=False)

print('Sample labels:')
for gid in list(active_ids)[:8]:
    n = panel_full[panel_full.cluster_id == gid]['count'].sum()
    print(f'  C{gid:03d}  {n:6,} patents   {cluster_labels[gid]}')

In [ ]:
# ── Save clustering artifacts ──────────────────────────────────────────────────
with open(OUT_DIR / 'cluster_centroids.pkl', 'wb') as f:
    pickle.dump(dict(cluster_centroids), f)
with open(OUT_DIR / 'cluster_titles.pkl', 'wb') as f:
    pickle.dump(dict(cluster_titles), f)

print('Clustering artifacts saved.')

---
## Part 2 — Feature Engineering

In [ ]:
ALL_FEATURES = [
    'log_count', 'count_share',
    'velocity', 'acceleration', 'jerk',
    'mom_4q', 'mom_8q',
    'roll_mean_4q', 'roll_std_4q', 'roll_mean_8q', 'roll_std_8q',
    'above_trend', 'macd', 'z_score',
    'global_rank_pctl',
    'log_cumsum',
    'sin_q', 'cos_q',
]
TARGET = 'log_count'

def make_features(panel: pd.DataFrame) -> pd.DataFrame:
    p = panel.copy().sort_values(['cluster_id', 'date'])
    p['log_count']   = np.log1p(p['count'])
    global_q         = p.groupby('date')['count'].transform('sum').replace(0, np.nan)
    p['count_share'] = p['count'] / global_q
    grp = p.groupby('cluster_id')['log_count']
    p['velocity']     = grp.diff().fillna(0)
    p['acceleration'] = p.groupby('cluster_id')['velocity'].diff().fillna(0)
    p['jerk']         = p.groupby('cluster_id')['acceleration'].diff().fillna(0)
    p['mom_4q'] = p.groupby('cluster_id')['log_count'].diff(4).fillna(0)
    p['mom_8q'] = p.groupby('cluster_id')['log_count'].diff(8).fillna(0)

    def roll(series, w, fn):
        return series.groupby(p['cluster_id']).transform(
            lambda x: getattr(x.rolling(w, min_periods=1), fn)()
        )
    p['roll_mean_4q'] = roll(p['log_count'], 4, 'mean')
    p['roll_std_4q']  = roll(p['log_count'], 4, 'std').fillna(0)
    p['roll_mean_8q'] = roll(p['log_count'], 8, 'mean')
    p['roll_std_8q']  = roll(p['log_count'], 8, 'std').fillna(0)
    p['above_trend']  = p['log_count'] - p['roll_mean_8q']
    p['macd']         = p['roll_mean_4q'] - p['roll_mean_8q']
    p['z_score']      = p['above_trend'] / (p['roll_std_8q'] + 1e-6)
    p['global_rank_pctl'] = p.groupby('date')['log_count'].rank(pct=True)
    p['log_cumsum']   = np.log1p(p.groupby('cluster_id')['count'].cumsum())
    p['sin_q']        = np.sin(2 * np.pi * p['quarter'] / 4)
    p['cos_q']        = np.cos(2 * np.pi * p['quarter'] / 4)
    return p.fillna(0)

df = make_features(panel_full)
print(f'Features: {len(ALL_FEATURES)}  |  Clusters: {df.cluster_id.nunique()}  |  Rows: {len(df):,}')

---
## Part 3 — Multi-Horizon Training

**Target:** `y = log_count[t+H] - log_count[t]` (growth in log-space over H quarters).
This directly optimises for what the leaderboard ranks on — acceleration, not absolute size.

**Horizons:** 2yr (8Q) and 3yr (12Q) as full GRU+LSTM models.
5yr (20Q) is a linear extrapolation shown on the dashboard — not a model claim.

**Baselines:** Persistence (predict 0 growth) and linear trend (extrapolate recent slope).

In [ ]:
# ── build_samples: delta target, anchor-point split, explicit TEST_END ─────────
def build_samples(df, split, horizon):
    Xs, ys, metas = [], [], []
    for cid, grp in df.groupby('cluster_id'):
        grp  = grp.sort_values('date').reset_index(drop=True)
        vals = grp[ALL_FEATURES].values
        n    = len(vals)
        for i in range(WINDOW, n - horizon + 1):
            anchor_row = grp.iloc[i - 1]
            ay, aq     = int(anchor_row['year']), int(anchor_row['quarter'])
            in_train   = (ay, aq) <= TRAIN_END
            in_val     = (not in_train) and (ay, aq) <= VAL_END
            in_test    = not in_train and not in_val and (ay, aq) <= TEST_END[horizon]
            if split == 'train' and not in_train: continue
            if split == 'val'   and not in_val:   continue
            if split == 'test'  and not in_test:  continue
            # y = growth in log-space over horizon quarters
            ys.append(
                float(grp.iloc[i + horizon - 1][TARGET]) - float(grp.iloc[i - 1][TARGET])
            )
            Xs.append(vals[i - WINDOW:i])
            metas.append({'cluster_id': cid, 'anchor_year': ay, 'anchor_q': aq})
    return (np.array(Xs, dtype=np.float32),
            np.array(ys, dtype=np.float32),
            metas)

# Quick sanity check on primary horizon
X_tr0, y_tr0, _ = build_samples(df, 'train', PRIMARY_H)
X_v0,  y_v0,  _ = build_samples(df, 'val',   PRIMARY_H)
print(f'Primary ({PRIMARY_H//4}yr) — train: {X_tr0.shape}  val: {X_v0.shape}')
print(f'y_train mean={y_tr0.mean():.3f}  std={y_tr0.std():.3f}  (near-0 expected)')

In [ ]:
from tensorflow import keras

T, F = WINDOW, len(ALL_FEATURES)

class TQDMEpochBar(keras.callbacks.Callback):
    """Clean per-epoch progress bar for Kaggle committed runs."""
    def on_train_begin(self, logs=None):
        self._pbar = tqdm(total=self.params['epochs'], desc='  epochs', unit='ep', leave=True)
    def on_epoch_end(self, epoch, logs=None):
        self._pbar.update(1)
        self._pbar.set_postfix({
            'val_loss': f"{logs.get('val_loss', 0):.4f}",
            'val_mae':  f"{logs.get('val_mae',  0):.4f}",
        })
    def on_train_end(self, logs=None):
        self._pbar.close()

def build_model(arch: str):
    inp = keras.Input(shape=(T, F), name='input')
    RNN = keras.layers.GRU if arch == 'gru' else keras.layers.LSTM
    x   = RNN(64, return_sequences=True, name=f'{arch}_1')(inp)
    x   = keras.layers.Dropout(0.2)(x)
    x   = RNN(32, name=f'{arch}_2')(x)
    x   = keras.layers.Dropout(0.2)(x)
    out = keras.layers.Dense(1, name='output')(x)
    m   = keras.Model(inp, out, name=arch)
    m.compile(optimizer=keras.optimizers.Adam(LR),
              loss=keras.losses.Huber(delta=1.0),
              metrics=['mae'])
    return m

def train_model(model, name, X_tr, y_tr, X_v, y_v):
    stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M')
    ckpt  = str(OUT_DIR / 'models' / f'{name}_{stamp}.keras')
    (OUT_DIR / 'models').mkdir(exist_ok=True)
    cbs = [
        TQDMEpochBar(),
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=10,
                                      restore_best_weights=True, verbose=1),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                          patience=5, verbose=0),
        keras.callbacks.ModelCheckpoint(ckpt, save_best_only=True, verbose=0),
    ]
    hist = model.fit(X_tr, y_tr,
                     validation_data=(X_v, y_v),
                     epochs=EPOCHS, batch_size=BATCH,
                     callbacks=cbs, verbose=0)
    return hist, ckpt

build_model('gru').summary()


In [ ]:
# ── Train GRU + LSTM for each horizon ─────────────────────────────────────────
horizon_models = {}

for H in tqdm(HORIZONS, desc='Horizons', unit='horizon'):
    hlabel = f'{H//4}yr'
    print(f'\n{"="*55}\nHorizon {hlabel} ({H}Q)\n{"="*55}')

    X_tr, y_tr, _ = build_samples(df, 'train', H)
    X_v,  y_v,  _ = build_samples(df, 'val',   H)
    X_te, y_te, _ = build_samples(df, 'test',  H)

    sc = StandardScaler()
    X_tr_sc = sc.fit_transform(X_tr.reshape(-1, F)).reshape(-1, T, F)
    X_v_sc  = sc.transform(X_v.reshape(-1, F)).reshape(-1, T, F)
    X_te_sc = sc.transform(X_te.reshape(-1, F)).reshape(-1, T, F)
    (OUT_DIR / 'models').mkdir(exist_ok=True)
    joblib.dump(sc, OUT_DIR / 'models' / f'scaler_{hlabel}.pkl')

    print(f'  Samples — train: {len(y_tr):,}  val: {len(y_v):,}  test: {len(y_te):,}')

    gru  = build_model('gru')
    print('  Training GRU ...')
    hist_gru, ckpt_gru   = train_model(gru,  f'gru_{hlabel}',  X_tr_sc, y_tr, X_v_sc, y_v)

    lstm = build_model('lstm')
    print('  Training LSTM ...')
    hist_lstm, ckpt_lstm = train_model(lstm, f'lstm_{hlabel}', X_tr_sc, y_tr, X_v_sc, y_v)

    gru_sp  = spearmanr(y_v, gru.predict(X_v_sc,  verbose=0).flatten())[0]
    lstm_sp = spearmanr(y_v, lstm.predict(X_v_sc, verbose=0).flatten())[0]
    best      = gru  if gru_sp >= lstm_sp else lstm
    best_arch = 'GRU' if gru_sp >= lstm_sp else 'LSTM'

    horizon_models[H] = {
        'gru': gru, 'lstm': lstm, 'best': best, 'best_arch': best_arch,
        'scaler': sc,
        'X_val': X_v,   'y_val': y_v,   'X_val_sc': X_v_sc,
        'X_test': X_te, 'y_test': y_te, 'X_test_sc': X_te_sc,
        'val_sp_gru': gru_sp, 'val_sp_lstm': lstm_sp,
        'ckpt_gru': ckpt_gru, 'ckpt_lstm': ckpt_lstm,
    }
    print(f'  Best: {best_arch}  val Spearman={max(gru_sp, lstm_sp):.4f}')


In [ ]:
# ── Training summary + baselines ───────────────────────────────────────────────
log_count_idx = ALL_FEATURES.index('log_count')
steps_arr     = np.arange(WINDOW, dtype=np.float32)
A_fit         = np.vstack([steps_arr, np.ones(WINDOW)]).T

for H, hd in horizon_models.items():
    hlabel  = f'{H//4}yr'
    y_v     = hd['y_val']
    X_v     = hd['X_val']

    # Baselines
    y_persist = np.zeros_like(y_v)
    log_hist  = X_v[:, :, log_count_idx]
    slopes    = np.linalg.lstsq(A_fit, log_hist.T, rcond=None)[0][0]
    y_trend   = slopes * H

    gru_pred  = hd['gru'].predict(hd['X_val_sc'],  verbose=0).flatten()
    lstm_pred = hd['lstm'].predict(hd['X_val_sc'], verbose=0).flatten()

    rows = []
    for pred, name in [(gru_pred, 'GRU'), (lstm_pred, 'LSTM'),
                       (y_trend, 'Linear trend'), (y_persist, 'Persistence (0)')]:
        sp, _ = spearmanr(y_v, pred)
        rmse  = float(np.sqrt(mean_squared_error(y_v, pred)))
        mae   = float(mean_absolute_error(y_v, pred))
        rows.append({'Model': name, 'Spearman': round(sp, 4),
                     'RMSE': round(rmse, 4), 'MAE': round(mae, 4)})

    print(f'\n── {hlabel} val metrics ──')
    print(pd.DataFrame(rows).to_string(index=False))

---
## Part 4 — Leaderboards

In [ ]:
# ── Model leaderboards (2yr and 3yr) ──────────────────────────────────────────
leaderboards = {}

for H, hd in horizon_models.items():
    hlabel  = f'{H//4}yr'
    rows    = []
    cluster_ids = df.cluster_id.unique()
    for cid in tqdm(cluster_ids, desc=f'Leaderboard {hlabel}', unit='cluster'):
        g = df[df.cluster_id == cid].sort_values('date')
        if len(g) < WINDOW: continue
        recent     = g.tail(WINDOW)[ALL_FEATURES].values.astype(np.float32)
        recent_sc  = hd['scaler'].transform(recent).reshape(1, T, F)
        pred_delta = float(hd['best'].predict(recent_sc, verbose=0)[0][0])
        last_log   = float(g.iloc[-1]['log_count'])
        rows.append({
            'cluster_id':         cid,
            'auto_label':         cluster_labels.get(cid, f'cluster_{cid}'),
            'last_log_count':     round(last_log, 3),
            'forecast_log_count': round(last_log + pred_delta, 3),
            'predicted_growth':   round(pred_delta, 3),
            'last_date':          str(g.iloc[-1]['date']),
        })
    lb = (pd.DataFrame(rows)
          .sort_values('predicted_growth', ascending=False)
          .reset_index(drop=True))
    lb.to_csv(OUT_DIR / f'leaderboard_{hlabel}.csv', index=False)
    leaderboards[H] = lb
    print(f'{hlabel} leaderboard — top 5:')
    print(lb[['auto_label', 'predicted_growth']].head(5).to_string(index=False))
    print()


In [ ]:
# ── 5yr linear extrapolation (not a model — explicit assumption) ──────────────
extrap_rows  = []
cluster_ids  = df.cluster_id.unique()
A_fit        = np.vstack([np.arange(WINDOW, dtype=np.float32), np.ones(WINDOW)]).T

for cid in tqdm(cluster_ids, desc='5yr extrapolation', unit='cluster'):
    g = df[df.cluster_id == cid].sort_values('date')
    if len(g) < WINDOW: continue
    log_hist = g.tail(WINDOW)['log_count'].values.astype(np.float32)
    slope    = np.linalg.lstsq(A_fit, log_hist, rcond=None)[0][0]
    extrap   = float(slope * 20)
    last_log = float(g.iloc[-1]['log_count'])
    extrap_rows.append({
        'cluster_id':              cid,
        'auto_label':              cluster_labels.get(cid, f'cluster_{cid}'),
        'last_log_count':          round(last_log, 3),
        'extrapolated_growth_5yr': round(extrap, 3),
        'last_date':               str(g.iloc[-1]['date']),
    })

lb_5yr = (pd.DataFrame(extrap_rows)
          .sort_values('extrapolated_growth_5yr', ascending=False)
          .reset_index(drop=True))
lb_5yr.to_csv(OUT_DIR / 'leaderboard_5yr_extrap.csv', index=False)
leaderboards['5yr_extrap'] = lb_5yr

print('5yr extrapolation (linear trend, not a model) — top 5:')
print(lb_5yr[['auto_label', 'extrapolated_growth_5yr']].head(5).to_string(index=False))


---
## Part 5 — Package Output

In [ ]:
# ── Zip everything into pipeline_output.zip ────────────────────────────────────
out_zip = OUT_DIR / 'pipeline_output.zip'

files_to_zip = [
    (OUT_DIR / 'cluster_panel.csv',       'clusters/cluster_panel.csv'),
    (OUT_DIR / 'cluster_labels.csv',      'clusters/cluster_labels.csv'),
    (OUT_DIR / 'cluster_centroids.pkl',   'clusters/cluster_centroids.pkl'),
    (OUT_DIR / 'cluster_titles.pkl',      'clusters/cluster_titles.pkl'),
    (OUT_DIR / 'leaderboard_2yr.csv',         'clusters/leaderboard_2yr.csv'),
    (OUT_DIR / 'leaderboard_3yr.csv',         'clusters/leaderboard_3yr.csv'),
    (OUT_DIR / 'leaderboard_5yr_extrap.csv',  'clusters/leaderboard_5yr_extrap.csv'),
]
for p in sorted((OUT_DIR / 'models').iterdir()):
    files_to_zip.append((p, f'models/{p.name}'))

with zipfile.ZipFile(out_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    for src, arcname in tqdm(files_to_zip, desc='Packaging', unit='file'):
        if src.exists():
            zf.write(src, arcname)
            tqdm.write(f'  + {arcname}  ({src.stat().st_size/1e6:.1f} MB)')
        else:
            tqdm.write(f'  ! missing: {src}')

print(f'\npipeline_output.zip  {out_zip.stat().st_size/1e6:.1f} MB')
print('Download from the Output tab, then:')
print('  unzip ~/Downloads/pipeline_output.zip -d /home/jin/Documents/GITHUB/patent-emerging-radar/data/processed/')
